# Spice to GDS converter
This code will be used in Chipathon 2026
Author : M Taufiqul Huda

In [ ]:
import sys
from pathlib import Path

root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180, sky130
from gdsfactory import Component
from glayout.util.port_utils import print_ports

from core import (
    clean_param, display_gds, display_component,
    set_pdk, spice_to_gds, llm_to_gds, generate_netlist_from_prompt,
    add_power_strips, add_double_guardring,
    run_drc, run_lvs, run_pex,
    run_ota_ac, run_comparator_tran, compare_pre_post,
    compare_comp_pre_post,
)

import os
import IPython.display
import gdstk

TEMP_DIR = os.getcwd()
GDS_PATH = os.path.join(TEMP_DIR, "out.gds")
SVG_PATH = os.path.join(TEMP_DIR, "out.svg")

In [ ]:
# ==========================================
# INPUT: SPICE Netlist — Clocked Comparator
# ==========================================

netlist_input = """
.lib "/foss/pdks/gf180mcuD/libs.tech/ngspice/sm141064.ngspice" typical
.subckt comp_clk vin_p vin_n clk vout_p vout_n vdd vss
M1 vout_p vin_p ntail vss nfet_03v3 W=5u L=1u
M2 vout_n vin_n ntail vss nfet_03v3 W=5u L=1u
M3 vout_p vout_n vdd vdd pfet_03v3 W=10u L=1u
M4 vout_n vout_p vdd vdd pfet_03v3 W=10u L=1u
M5 ntail clk vss vss nfet_03v3 W=10u L=1u
M6 vout_p clk vdd vdd pfet_03v3 W=5u L=1u
M7 vout_n clk vdd vdd pfet_03v3 W=5u L=1u
.ends
"""

In [ ]:
# ==========================================
# RUN PIPELINE: SPICE -> GDS
# ==========================================

# result = spice_to_gds(netlist_input, mode="analog", add_labels=True)
# result.write_gds(GDS_PATH)
# print(f"[DONE] Layout written to {GDS_PATH}")
# display_component(result, scale=2)

# ==========================================
# Optional: DRC / LVS / PEX
# ==========================================
# Set environment (adjust paths for your setup):
import os
os.environ["PDK_ROOT"] = "/foss/pdks"
os.environ["PDK"] = "gf180mcuD"
os.environ["PDKPATH"] = "/foss/pdks/gf180mcuD"

# Run with post-layout verification:
result = spice_to_gds(netlist_input, mode="analog", add_labels=True,
                      run_checks=True)
result.write_gds("out.gds")

# Or run checks individually (GDS written as {cell}.gds):
drc = run_drc("comp_clk.gds", cell_name="comp_clk", engine="magic")
print(drc["log"])
# lvs = run_lvs("comp_clk.gds", netlist_content=netlist_input,
#                cell_name="comp_clk")
# pex = run_pex("comp_clk.gds", cell_name="comp_clk", mode=3)


In [ ]:
# ==========================================
# SIMULATION: Comparator Transient
# ==========================================

# Transient (step response, 10mV differential):
tran = run_comparator_tran(netlist_input, "comp_clk",
                           vdd=3.3, vcm=1.65,
                           clk_period=20e-9, tstop=50e-9)
td = tran["tdelay"] * 1e9  # seconds → ns
print(f'[COMP] Swing={tran["vout_low"]:.2f}..{tran["vout_high"]:.2f}V  tpd={td:.2f} ns')

# Offset measurement (slow ramp on vin_p, vin_n = VCM):
off = run_comparator_tran(netlist_input, "comp_clk",
                          vdd=3.3, vcm=1.65,
                          clk_period=20e-9, tstop=50e-9,
                          measure_offset=True)
vos_uv = off["vos"] * 1e6  # V → uV
print(f"[COMP] Input Offset = {vos_uv:.1f} uV")


In [ ]:
# ==========================================
# POST-SIMULATION: PEX vs Schematic
# ==========================================

# 1. Generate PEX (C-coupled):
pex = run_pex("comp_clk.gds", cell_name="comp_clk", mode=2)
#
# 2. Compare schematic vs PEX:
cmp = compare_comp_pre_post(netlist_input, pex["pex_path"], "comp_clk",
                            vdd=3.3, vcm=1.65,
                            clk_period=20e-9, tstop=50e-9)
#
for k in ("tdelay", "vout_high", "vout_low"):
    pre_v = cmp["pre"].get(k)
    post_v = cmp["post"].get(k)
    ps = f"{pre_v*1e9:.2f}ns" if pre_v and "delay" in k else f"{pre_v:.4g}" if pre_v else "N/A"
    pp = f"{post_v*1e9:.2f}ns" if post_v and "delay" in k else f"{post_v:.4g}" if post_v else "N/A"
    print(f"{k:15s}  pre={ps:>12s}  post={pp:>12s}")
#
# Note: PEX extraction quality depends on layout complexity.
# For ng=1 designs, compare_comp_pre_post() maps pin order correctly
# and strips .option scale / .lib / ng= from the PEX netlist.
# Full RC (mode=3) requires advanced simulator convergence settings.
